In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json, yaml
from glob import glob
from sklearn.model_selection import train_test_split

# --- 1. DEFINE PATHS ON DRIVE ---
base_drive_path = '/content/drive/MyDrive/TACO_YOLO'

print("Setting up folders on Google Drive...")
for d in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    os.makedirs(os.path.join(base_drive_path, d), exist_ok=True)

Mounted at /content/drive
Setting up folders on Google Drive...


In [ ]:
# --- 2. FAST EXTRACTION (ON LOCAL CPU STORAGE) ---
print("Extracting data locally for speed...")
!mkdir -p /content/temp_data
!unzip -q -o "/content/drive/MyDrive/archive (1).zip" -d "/content/temp_data"

# --- 3. SPLIT AND MOVE TO DRIVE ---
source_path = '/content/temp_data/data'
all_images = glob(os.path.join(source_path, 'batch_*/**/*.jpg'), recursive=True) + \
             glob(os.path.join(source_path, 'batch_*/**/*.JPG'), recursive=True)

train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

def move_to_drive_uniquely(files, subset):
    for f in files:
        parts = f.split('/')
        batch_name = next((p for p in reversed(parts) if 'batch' in p), "batch_unknown")
        new_name = f"{batch_name}_{parts[-1]}"
        shutil.copy(f, os.path.join(base_drive_path, subset, 'images', new_name))

print("Moving images to Drive... this may take a few minutes.")
move_to_drive_uniquely(train_files, 'train')
move_to_drive_uniquely(val_files, 'val')

Extracting data locally for speed...
Moving images to Drive... this may take a few minutes.


In [ ]:
# --- 4. CONVERT ANNOTATIONS DIRECTLY TO DRIVE ---
json_path = '/content/temp_data/data/annotations.json'
with open(json_path, 'r') as f:
    taco_data = json.load(f)

image_info = {img['id']: img for img in taco_data['images']}
category_mapping = {cat['id']: i for i, cat in enumerate(taco_data['categories'])}

print("Converting labels to YOLO format on Drive...")
for ann in taco_data['annotations']:
    meta = image_info.get(ann['image_id'])
    if not meta: continue

    unique_fname = meta['file_name'].replace('/', '_')
    img_name_only = os.path.splitext(unique_fname)[0]

    if os.path.exists(f'{base_drive_path}/train/images/{unique_fname}'):
        lbl_path = f'{base_drive_path}/train/labels/{img_name_only}.txt'
    elif os.path.exists(f'{base_drive_path}/val/images/{unique_fname}'):
        lbl_path = f'{base_drive_path}/val/labels/{img_name_only}.txt'
    else: continue

    yolo_id = category_mapping[ann['category_id']]
    dw, dh = 1./meta['width'], 1./meta['height']
    x, y, w, h = ann['bbox']
    yolo_row = f"{yolo_id} {(x + w/2)*dw:.6f} {(y + h/2)*dh:.6f} {w*dw:.6f} {h*dh:.6f}\n"

    with open(lbl_path, 'a') as f:
        f.write(yolo_row)

Converting labels to YOLO format on Drive...


In [ ]:
# --- 5. CREATE YAML ON DRIVE ---
taco_names_dict = {i: cat['name'] for i, cat in enumerate(taco_data['categories'])}
taco_config = {
    'path': base_drive_path,
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(taco_names_dict),
    'names': taco_names_dict
}

with open(f'{base_drive_path}/taco.yaml', 'w') as f:
    yaml.dump(taco_config, f, sort_keys=False)

print(f"DONE! Your dataset is ready at: {base_drive_path}")

DONE! Your dataset is ready at: /content/drive/MyDrive/TACO_YOLO


In [ ]:
# Changing the runtime type (CPU -> GPU)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00


In [ ]:
# Copy the processed data from Drive to local /content (very fast)
!cp -r /content/drive/MyDrive/TACO_YOLO/train /content/train
!cp -r /content/drive/MyDrive/TACO_YOLO/val /content/val

In [ ]:
from ultralytics import YOLO

# Loading the model
model = YOLO("yolo26n.pt")

# Start training using
results = model.train(
    data='/content/drive/MyDrive/TACO_YOLO/taco.yaml', # New path
    epochs=100,
    batch=64,
    patience=20,
    imgsz=416,
    device=0,
    plots=True,
    workers=4,
    optimizer='MuSGD',
    project='/content/drive/MyDrive/TACO_YOLO/runs' # Save the results to Drive
)


Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TACO_YOLO/taco.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=MuSGD, overlap_mask=True, patience=20,